# Dataset Genration for Q1

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

transactions_data = [
    ("t1", "u1", "p1", "$120.50", "USD", "2025-02-01 10:15:00"),
    ("t2", "u2", "p2", "$200.00", "USD", "01-02-2025 11:20:00"),  
    ("t3", None, "p3", "$50.00", "USD", "2025/02/01 12:00:00"), 
    ("t4", "u3", "p4", "-$30.00", "USD", "2025-02-01 13:00:00"), 
    ("t5", "u1", "p5", "$75.00", "USD", "2025-02-01 14:00:00"),
    ("t5", "u1", "p5", "$75.00", "USD", "2025-02-01 14:05:00"), 
    ("t6", "u4", "p2", "$500.00", "USD", "2025-02-02 09:00:00"),
    ("t7", "u2", "p3", "$300.00", "USD", "2025-02-02 10:30:00"),
    ("t8", "u3", "p1", "$150.00", "USD", "2025-02-02 11:00:00"),
]

transactions_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
])

transactions_df = spark.createDataFrame(transactions_data, schema=transactions_schema)

users_data = [
    ("u1", "uk", "2024-01-01"),
    ("u2", "Us", "2024-02-01"),
    ("u3", "IN", "2024-03-01"),
    ("u3", "in", "2024-03-02"),  
    ("u4", None, "2024-04-01"),  
]

users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", StringType(), True),
])

users_df = spark.createDataFrame(users_data, schema=users_schema)




In [0]:
#CHANGE BELOW LOCATION TO SAVE THIS DATA 
target = "/Volumes/practice/bronze/raw_ingestion/csv"

#####
transactions_df.write.mode("overwrite").option("header", True).csv(f"{target}/transactions_csv")

users_df.write.mode("overwrite").option("header", True).csv(f"{target}/users_csv")


# Pair Programming Task — End-to-End PySpark Pipeline
---
## Scenario

You are given two datasets:

### 📁 `transactions.csv`

```
transaction_id (string)
user_id (string)
product_id (string)
amount (string)
currency (string)
transaction_timestamp (string)
```

Problems:

* amount contains currency symbols (e.g. "$120.50")
* timestamp format inconsistent
* duplicates exist
* some null user_id rows
* some negative amounts

---

### 📁 `users.csv`

```
user_id (string)
country (string)
signup_date (string)
```

Problems:

* duplicate users
* country casing inconsistent
* null values

---




## Step 1 — Read Data

* Read both datasets
* Define schema explicitly
* Handle malformed rows safely

---

In [0]:
df_t = spark.read.options(header=True).csv("/Volumes/practice/bronze/raw_ingestion/csv/transactions_csv/")
df_u = spark.read.options(header=True).csv("/Volumes/practice/bronze/raw_ingestion/csv/users_csv/")

In [0]:
df_t.display()
df_u.display()

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_u = df_u.withColumn("last_updated_ts", current_timestamp())\
    .withColumn("file_path", col("_metadata.file_path"))

df_t = df_t.withColumn("last_updated_ts", current_timestamp())\
    .withColumn("file_path", col("_metadata.file_path"))


In [0]:
# Write datframe to bronze layer tables 
df_t.write.mode("overwrite").format("delta").saveAsTable("practice.bronze.brz_transactions")
df_u.write.mode("overwrite").format("delta").saveAsTable("practice.bronze.brz_users")

## Step 2 — Clean Data

### Transactions:

* Remove null user_id
* Remove negative amounts
* Clean amount → cast to double
* Standardise timestamp → proper timestamp type
* Deduplicate based on latest transaction_timestamp per transaction_id



### Users:

* Deduplicate based on latest signup_date
* Standardise country to uppercase

---

In [0]:
df_t = spark.read.table("practice.bronze.brz_transactions")
df_u = spark.read.table("practice.bronze.brz_users")
df_t.display()
df_u.display()

In [0]:
# drop null rows
df_t = df_t.dropna(subset=['user_id'])
df_u = df_u.dropna(subset=['country'])


In [0]:
from pyspark.sql.functions import to_date, col, initcap, upper

# Convert signup_date from string type to date type
df_u = df_u.withColumn("signup_date", to_date(col("signup_date")))
# Capitalize country names
df_u = df_u.withColumn("country", upper(col("country")))

df_u.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# "Deduplicate rows based on latest signup_date"
window_spec_u = Window.partitionBy("user_id")\
    .orderBy(col("signup_date").desc())
df_u = df_u.withColumn("rn", row_number().over(window_spec_u)).filter(col("rn") == 1).drop("rn")

df_u.display()

In [0]:
from pyspark.sql.functions import regexp_replace, col

# Remove $ and negative sign, then cast to double to change datatype from string
df_t = df_t.withColumn("amount", regexp_replace(col("amount"), r"[\$-]", "").cast("double"))
df_t.display()

In [0]:
from pyspark.sql.functions import expr

# Handle multiple timestamp formats using sql expr function
df_t = df_t.withColumn(
    "transaction_timestamp",
    expr("""
        CASE
            WHEN transaction_timestamp RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'
                THEN to_timestamp(transaction_timestamp, 'yyyy-MM-dd HH:mm:ss')
            WHEN transaction_timestamp RLIKE '^[0-9]{2}-[0-9]{2}-[0-9]{4}'
                THEN to_timestamp(transaction_timestamp, 'dd-MM-yyyy HH:mm:ss')
            WHEN transaction_timestamp RLIKE '^[0-9]{4}/[0-9]{2}/[0-9]{2}'
                THEN to_timestamp(transaction_timestamp, 'yyyy/MM/dd HH:mm:ss')
        END
    """)
)
df_t.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

'''window_spec_t = Window.partitionBy("transaction_id")\
    .orderBy(col("transaction_timestamp").desc())'''
# "Deduplicate rows based on latest transaction_timestamp"
df_t = df_t.withColumn(
    "rn", row_number().over(
                        Window.partitionBy("transaction_id")\
                                .orderBy(col("transaction_timestamp").desc()))
).filter(col("rn") == 1).drop("rn")

df_t.display()


In [0]:
# Select all columns except last two metadata columns before writing to silver layer

cols_u = df_u.columns[:-2]
cols_t = df_t.columns[:-2]

df_u = df_u.select(*cols_u)
df_t = df_t.select(*cols_t)

df_u.display()
df_t.display()

In [0]:
# Write dataframe to silver layer tables
df_u.write.mode("overwrite").saveAsTable("practice.silver.slv_users")
df_t.write.mode("overwrite").saveAsTable("practice.silver.slv_transactions")

## Step 3 — Transform

1. Join transactions with users
2. Calculate:

   * Daily revenue per country
3. Return:

   * transaction_date
   * country
   * total_revenue

---


In [0]:
# Read dataframes from silver layer tables
df_u = spark.read.table("practice.silver.slv_users")
df_t = spark.read.table("practice.silver.slv_transactions")


In [0]:
# Join transactions with users 
df = df_t.join(df_u, on="user_id", how="inner")

df.display()

In [0]:
# Calculate daily revenue per country
from pyspark.sql.functions import col, desc

revenue_df = df.groupBy("country", col("transaction_timestamp").cast("date").alias("transaction_date"))\
    .sum("amount")\
        .withColumnRenamed("sum(amount)", "total_revenue")\
            .orderBy("country","transaction_date")

revenue_df.display()

## Step 4 — Advanced Requirement

For each country and day:

* Return top 3 users by revenue

Output:

```
transaction_date
country
user_id
total_user_revenue
rank
```

---

In [0]:
from pyspark.sql.functions import row_number, col, sum

# Calculate total revenue per user per country per day
user_revenue_df = df.groupBy(
    col("country"),
    col("user_id"),
    col("transaction_timestamp").cast("date").alias("transaction_date")
).agg(sum("amount").alias("total_user_revenue"))\
    .orderBy("country","transaction_date")
user_revenue_df.display()

In [0]:
from pyspark.sql.window import Window

# Window spec for ranking users by revenue per country and day
window_spec_rev = Window.partitionBy("country", "transaction_date")\
    .orderBy(col("total_user_revenue").desc())
    
# Add rank and filter top 3
top_users_df = user_revenue_df.withColumn(
    "rank", row_number().over(window_spec_rev)
).filter(col("rank") <= 3)

top_users_df.display()


## Step 5 — Performance Discussion

After coding they will ask:

* What causes shuffle here?
* Where would you broadcast?
* How would you optimise for 1 billion rows?
* How would you partition when writing?
* How would you productionise this?

---